## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://private-user-images.githubusercontent.com/181020547/465824799-12a2371b-8be2-4219-9b48-90503eb43c69.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NjAwNDgyMzcsIm5iZiI6MTc2MDA0NzkzNywicGF0aCI6Ii8xODEwMjA1NDcvNDY1ODI0Nzk5LTEyYTIzNzFiLThiZTItNDIxOS05YjQ4LTkwNTAzZWI0M2M2OS5wbmc_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1BS0lBVkNPRFlMU0E1M1BRSzRaQSUyRjIwMjUxMDA5JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MTAwOVQyMjEyMTdaJlgtQW16LUV4cGlyZXM9MzAwJlgtQW16LVNpZ25hdHVyZT1iYTRmYTAzYjkzYjA2MGE4ZTZlYjQ4ODU1OWIwY2VlZWU0Mzk0YzdmMjQ1YTlhMDMyNmI3NWNlZTQxNDdlZGViJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCJ9.a8477QD1J4Lrmys7jB8gt_H5pdiKBsKsu3npEqZjEpo)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

#### ✅ Answer:
- Each state handles a specific responsibility level with clear boundaries
- Easier to maintain and debug.
- Can help with memory management, to prevent memory bloat, and to handle token limits
- Can help provide clearer data flows between components.
- Can help isolate errors, so that failures in one researcher don't affect others. 

## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

#### ✅ Answer:

Advantages: 
- Clean separation of concerns - Notebook focuses on workflow demonstration while library handles implementation details 

- Better maintainability - Single source of truth for components, easier debugging, and independent testing of modules

- Better scalability - Reusable components

Disadvantages:

- Debugging challenges - Errors occur in library code making them harder to trace, and stack traces reference library files instead of notebook

- Reduced hands-on learning - Components feel like "black boxes" and you can't see implementation being built or easily experiment with modifications


## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [16]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [17]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [18]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with analyzing the NBER working paper "How People Use ChatGPT." You've provided the PDF content and requested insights on three key areas:

1. Main findings about how people are using AI
2. Most common use cases  
3. Trends and patterns from the data

The document contains comprehensive data about ChatGPT usage patterns, user demographics, work vs. non-work usage, conversation topics, and growth trends from November 2022 through July 2025. I will now analyze this research and provide you with detailed insights covering all three areas you've requested.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER working paper "How People Use ChatGPT" (Working Paper No. 34255, September 2025) by Aaron Chatterji, Thomas Cunningham, David J. Deming, Zoe Hitzig, Christopher Ong, Carl Yan Shan, and Kevin Wadman. Please provide detailed insights on three spe


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Comprehensive Analysis of "How People Use ChatGPT": A Deep Dive into AI Adoption and Usage Patterns

The NBER working paper "How People Use ChatGPT" represents the most comprehensive study to date of how consumers actually use large language model chatbots in their daily lives. This research, conducted by a team including Aaron Chatterji, David J. Deming, and colleagues from Harvard, Duke, and OpenAI, analyzes actual usage data from ChatGPT's meteoric rise from launch to becoming a platform used by 10% of the world's adult population [1].

## Main Findings: User Demographics and Adoption Patterns

### Unprecedented Growth Trajectory

ChatGPT's adoption has been historically unprecedented. Launching in November 2022, the platform reached one million registered users within just five days [1]. By early November 2023, less than one year after release, ChatGPT had achieved 100 million weekly active users. The platform's growth continued at an extraordinary pace, with weekly active users doubling every 7-8 months since reaching that initial 100 million milestone [1].

By July 2025, ChatGPT had reached 700 million users sending 18 billion messages weekly, representing approximately 10% of the global adult population. This scale is further emphasized by the fact that users were sending more than 2.6 billion messages per day by June 2025—equivalent to over 30,000 messages per second [1]. The intensity of usage has also increased dramatically, with total message volume growing 5.8 times in the last year while user volume grew 3.2 times, indicating that existing users are engaging more deeply with the platform [1].

### Dramatic Shift from Work to Non-Work Usage

One of the study's most significant findings is the fundamental shift in how people use ChatGPT over time. In June 2024, non-work usage accounted for 53% of all messages, but by June 2025, this had grown to over 70% of all usage—specifically 73% [1][2]. This represents a complete reversal of initial expectations that AI chatbots would primarily serve professional purposes.

The growth in non-work messages has substantially outpaced work-related usage, even though both categories have grown continuously. By July 2025, only 27% of usage was work-related [2]. Importantly, this shift is primarily driven by changing usage patterns within existing user cohorts rather than changes in the composition of new users joining the platform. This suggests that as users become more familiar with ChatGPT's capabilities, they increasingly find value in applying it to personal, non-professional contexts [1].

### Narrowing Demographics Gaps

The study reveals remarkable changes in user demographics over ChatGPT's brief history. Initially, early adopters were overwhelmingly male, with more than 80% of weekly active users having typically male first names at launch [1]. However, this gender gap has narrowed dramatically. By June 2025, the user base had reached near parity, with 52% of active users having typically female first names—a complete reversal from the platform's early days [1].

Age distribution shows that nearly half (46%) of all adult messages come from users under 26, though work usage increases with age [2]. This suggests that younger users are driving much of the platform's non-work growth, using ChatGPT for educational, creative, and personal purposes.

### Global Expansion and Income Distribution

Perhaps most remarkably, the study finds higher growth rates in lower-income countries, with geographic usage gaps narrowing significantly. Middle-income countries have experienced 5-6 times growth compared to 3 times growth in the richest countries [1]. Countries as economically diverse as Brazil, South Korea, and the United States now show similar usage rates despite having vastly different GDP per capita ($10,000, $34,000, and $86,000 respectively) [1]. This suggests that ChatGPT's value proposition transcends economic boundaries and that the technology is becoming truly democratized globally.

## Most Common Use Cases: The Three Dominant Categories

### The 80% Rule: Three Categories Dominate Usage

The researchers' automated classification system revealed that nearly 80% of all ChatGPT conversations fall into three broad categories: Practical Guidance (29%), Seeking Information (24%), and Writing (24%) [1][2]. This concentration suggests that while ChatGPT enables a wide variety of use cases, most users gravitate toward a relatively focused set of core applications.

### Practical Guidance: The Leading Use Case

At 29% of all usage, Practical Guidance represents the single most common way people interact with ChatGPT [2]. This category encompasses tutoring and teaching interactions, how-to advice across various topics, and creative ideation. The dominance of this category reveals that users frequently turn to ChatGPT as a knowledgeable advisor or mentor figure, seeking guidance on both simple and complex problems.

The popularity of Practical Guidance reflects ChatGPT's unique ability to provide personalized, contextual advice. Unlike traditional search engines that return lists of links, ChatGPT can engage in back-and-forth dialogue to understand specific circumstances and provide tailored recommendations. This conversational aspect appears to be particularly valuable for users seeking guidance rather than just information.

### Seeking Information: The Search Engine Alternative

Representing 24% of usage, Seeking Information includes activities like searching for information about people, current events, products, and recipes [2]. This category appears to function as a very close substitute for traditional web search, but with the added benefit of natural language interaction and the ability to ask follow-up questions.

The substantial share of information-seeking behavior suggests that many users view ChatGPT as an alternative to Google or other search engines. However, the conversational format allows for more nuanced queries and the ability to refine searches through dialogue, potentially making information discovery more efficient and user-friendly.

### Writing: The Work-Dominant Category

Writing, also at 24% of overall usage, shows the most pronounced difference between work and non-work contexts. This category includes automated production of emails, documents, and other communications, as well as editing, critiquing, summarizing, and translating text provided by users [2]. Writing accounts for 40% of work-related messages, making it the dominant work use case [2].

Particularly interesting is that approximately two-thirds of all Writing messages ask ChatGPT to modify existing user text—editing, critiquing, translating, or improving it—rather than creating entirely new content from scratch [2]. This suggests that users often view ChatGPT as a sophisticated editing and enhancement tool rather than just a content generator.

### Work vs. Non-Work Context Differences

The distribution of these categories varies significantly between work and non-work contexts. Work usage is more common among educated users in highly-paid professional occupations, with Writing dominating professional use cases [1][2]. The researchers mapped usage to O*NET work activities and found that the top activities include getting information, interpreting information, documenting/recording, making decisions, solving problems, and thinking creatively—essentially the core activities of knowledge work [2].

## Emerging Trends and Patterns

### User Intent Classification Reveals Behavioral Patterns

The study classified messages into three intent categories: "Asking" (seeking information, judgment, or perspective), "Doing" (generating outputs like drafts, code, or slides), and "Expressing" (reflecting, reacting, or playing) [2]. Across all consumer usage, the distribution is approximately 49% Asking, 40% Doing, and 11% Expressing.

For work-related usage specifically, the pattern shifts significantly, with 56% of work queries classified as "Doing" versus 35% as "Asking" [2]. This suggests that in professional contexts, users are more likely to use ChatGPT to actually perform tasks rather than just seek advice or information. Interestingly, Asking messages receive higher ratings from both automated quality classifiers and direct user feedback compared to Doing messages, suggesting that ChatGPT may currently be more effective at providing guidance than at task execution [2].

### Evolution of Usage Intensity Across User Cohorts

The study reveals fascinating patterns in how different user cohorts have evolved their ChatGPT usage over time. All user cohorts showed similar patterns: relatively flat usage through 2024, followed by substantial increases beginning in early 2025 [1]. This suggests that major improvements to the platform occurred around early 2025, likely including the introduction of new models or features that significantly enhanced user experience.

Early adopters who signed up in Q1 2023 are sending 40% more messages per day in July 2025 than they were two years earlier. Even more dramatically, later cohorts who joined in Q3-Q4 2024 are sending nearly twice as many messages as they were less than a year ago [1]. This pattern indicates that ChatGPT's value proposition continues to strengthen for users over time, rather than following a typical pattern of initial enthusiasm followed by declining engagement.

### Surprising Findings About Programming and Social Usage

Contrary to popular assumptions and other studies, the research found that computer programming represents only 4.2% of all ChatGPT messages [2]. This is significantly lower than other studies and platforms—for comparison, 33% of work-related Claude conversations are programming-related [2]. This suggests that ChatGPT's user base is much more mainstream and diverse than the developer-heavy early adopter communities that often drive initial AI tool adoption.

Similarly, companionship or social-emotional use represents just 2.3% of conversations [2]. This finding contradicts popular media narratives about AI chatbots serving primarily as social companions or therapists. Instead, the data shows users predominantly engaging with ChatGPT for practical, task-oriented purposes.

### Geographic and Economic Democratization

The study reveals that ChatGPT usage is becoming increasingly democratized globally. Growth has been fastest in lower-income countries, with middle-income nations showing 5-6 times growth compared to 3 times growth in the richest countries [1]. This trend suggests that AI technology is not becoming a privilege of wealthy nations but is instead providing value across diverse economic contexts.

The fact that countries with vastly different economic profiles now show similar usage rates indicates that ChatGPT's value proposition transcends traditional economic barriers. This democratization could have profound implications for global knowledge access and economic development, particularly if the trend continues.

### Implications for Economic Value Creation

The research concludes that ChatGPT provides economic value primarily through decision support, which is especially important in knowledge-intensive jobs [1][2]. This finding aligns with broader economic theories about AI's impact on cognitive work. Rather than replacing human decision-makers, ChatGPT appears to be augmenting human decision-making capabilities by providing rapid access to information, alternative perspectives, and analytical support.

The substantial growth in non-work usage also suggests significant consumer surplus creation outside traditional economic productivity measures. The researchers note that while most economic analysis of AI focuses on productivity gains in paid work, the impact on home production and personal activities may be on a similar scale or even larger [1]. This has important implications for how economists and policymakers should think about AI's societal benefits.

### Methodological Innovation in AI Research

The study represents a methodological breakthrough in AI adoption research by addressing three major problems that typically limit such studies: small sample sizes, short time horizons, and reliance on self-reported usage data [3]. By analyzing actual usage logs across 2.5 years covering billions of interactions, the research provides an unprecedented view into how people actually use AI in practice, not how they think they use it or how researchers expect them to use it [3].

The privacy-preserving automated classification methodology used in this study—where over 1 million messages were analyzed without any human inspection—sets a new standard for conducting large-scale research on sensitive user data while maintaining privacy protections [2]. This approach could serve as a model for future research in the AI space where balancing research insights with user privacy is critical.

The study's findings challenge many assumptions about AI adoption and usage patterns while revealing the true scale and diversity of how people integrate AI tools into their daily lives. As ChatGPT and similar tools continue to evolve, this research provides crucial baseline data for understanding one of the most significant technological adoption stories in human history.

### Sources

[1] How People Use ChatGPT - by David Deming: https://forklightning.substack.com/p/how-people-use-chatgpt
[2] How People Use ChatGPT | NBER: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf
[3] How People Use ChatGPT: Part 1, Methodology and Scope | yamz8: https://www.yamz8.com/blog/how-people-use-chatgpt-part-1-methodology-and-scope


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

#### Answer: 

In [19]:
import time

# Test configurations 
configs = {
    "baseline": {
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        "search_api": "tavily",
        "allow_clarification": True
    },
    
    "more_parallel": {
        "max_concurrent_research_units": 5,  # Changed from 1
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        "search_api": "tavily",
        "allow_clarification": True
    },
    
    "deeper_research": {
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 6,  # Changed from 2
        "max_react_tool_calls": 8,      # Changed from 3
        "search_api": "tavily",
        "allow_clarification": True
    }
}

# Simple test question
test_question = "What are the latest AI trends in 2024?"

In [20]:
async def test_config(config_name, config_settings):
    print(f"\nTesting: {config_name}")
    print("-" * 40)
    
    # Start timer
    start_time = time.time()
    
    # Create config with settings
    test_config = {
        "configurable": {
            "research_model": "anthropic:claude-sonnet-4-20250514",
            "search_api": "tavily",
            "thread_id": str(uuid.uuid4()),
            **config_settings  # Add the test settings
        }
    }
    
    # Run research 
    await run_research()  
    
    # End timer
    end_time = time.time()
    print(f"⏱️  Time: {end_time - start_time:.1f} seconds")
    print(f"✅ {config_name} completed")

In [21]:
# Test each configuration
for name, settings in configs.items():
    await test_config(name, settings)

print("\n" + "="*50)
print("All tests completed!")
print("="*50)


Testing: baseline
----------------------------------------
Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with your analysis. You've provided a comprehensive NBER working paper titled "How People Use ChatGPT" and requested insights on: (1) main findings about AI usage patterns, (2) most common use cases, and (3) emerging trends from the data. The document contains detailed statistics, classifications, and analysis of ChatGPT usage from November 2022 through July 2025. I will now analyze this research paper and provide you with a comprehensive report covering all three areas you've specified.

Node: write_research_brief

Research Brief Generated:
I need you to analyze the NBER working paper "How People Use ChatGPT" by Chatterji et al. (2025) and provide comprehensive insights on three specific areas: (1) What are the main findings about how people are using AI based on this research, including adoption patterns, demographic trends, and


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# How People Use ChatGPT: Comprehensive Analysis from NBER Research (2025)

The NBER working paper "How People Use ChatGPT" by Chatterji et al. (2025) represents the largest and most comprehensive study of AI chatbot usage to date, analyzing 1.5 million conversations through a privacy-preserving methodology. This research provides unprecedented insights into how artificial intelligence is being adopted and used across global populations, revealing patterns that challenge conventional assumptions about AI's primary applications.

## Main Findings: AI Usage Patterns and Demographic Evolution

### Unprecedented Adoption and Scale

ChatGPT's growth trajectory represents an unprecedented pace of technology adoption. Launched as a research preview on November 30, 2022, the platform reached one million users within five days and has since scaled to approximately 700 million weekly active users by July 2025, representing roughly 10% of the world's adult population [1]. The platform now processes over 2.6 billion messages daily as of June 2025, with message volume growing 5.8x in just the past year [1][2].

This scale of adoption has no historical precedent for a new technology, positioning ChatGPT as likely the largest mass-market chatbot globally. The research documents daily message volume jumping from 451 million in June 2024 to 2.63 billion in June 2025, representing a nearly six-fold increase [3][4].

### Demographic Transformation and Equality Trends

The study reveals dramatic shifts in ChatGPT's user demographics that challenge early assumptions about AI adoption patterns. Initially dominated by highly educated men from wealthy countries, with over 80% male users at launch, usage has rapidly equalized across gender lines. By July 2025, the platform reached near gender parity with 52% female users, and by June 2025, weekly active users were actually slightly more likely to have typically feminine first names [1][2].

This demographic evolution extends beyond gender. Geographic adoption has broadened substantially, with middle-income countries showing 5-6x growth compared to 3x growth in the richest countries. Most notably, growth in the lowest-income countries has been more than four times that of highest-income countries, suggesting AI tools are democratizing access to advanced capabilities globally [1][3].

Age demographics also reveal interesting patterns, with nearly half of all adult messages coming from users under 26, though age gaps have narrowed somewhat in recent months [2]. This concentration among younger users suggests generational differences in AI adoption and comfort levels.

### User Retention and Engagement Evolution

Perhaps most surprisingly, the research documents increasing user engagement across all cohorts regardless of signup date. All user cohorts experienced relatively flat usage through 2024 followed by substantial increases beginning in early 2025. Early adopters now send 40% more messages daily than two years earlier, while recent users nearly doubled their daily message volume within a year [1].

This pattern suggests significant improvements in ChatGPT's capabilities or user-friendliness, as existing users are finding increasing value rather than experiencing typical technology fatigue. The consistent pattern across all signup cohorts indicates this isn't merely about user composition changes but reflects genuine improvements in utility.

## Most Common Use Cases and Usage Classifications

### The Big Three: Core Usage Categories

The research's automated classification system reveals that nearly 80% of all ChatGPT conversations fall into three primary categories: Practical Guidance, Seeking Information, and Writing [2][3]. This concentration suggests that while AI capabilities are broad, actual usage patterns are more focused than might be expected.

**Practical Guidance** emerges as the most common use case, maintaining a consistent 29% of overall usage. This category encompasses tutoring and teaching (10.2% of all messages), general how-to advice (8.5% of all messages), and creative ideation. The stability of this category suggests it represents a core value proposition that users consistently find valuable [2][4].

**Seeking Information** has shown the most dramatic growth, expanding from 14% to 24% of all usage between July 2024 and June 2025. The research notes this appears to be "a very close substitute for web search," suggesting ChatGPT is capturing search engine market share by providing more conversational and contextual information retrieval [2][4].

**Writing** has remained substantial but declining as a percentage, dropping from 36% to 24% of usage over the same period. Importantly, about two-thirds of all Writing messages ask ChatGPT to modify existing user text (editing, critiquing, translating) rather than creating new content from scratch. This suggests users primarily view ChatGPT as a sophisticated editing and enhancement tool rather than a replacement for human creativity [4].

### Work vs. Non-Work Usage Patterns

One of the study's most significant findings challenges assumptions about AI's primary value proposition. Non-work-related messages have grown from 53% to more than 70% of all usage between June 2024 and June 2025 [3][4]. This shift occurred primarily due to changing usage patterns within existing user cohorts rather than changes in user composition.

The research reveals that 70% of ChatGPT use is now personal rather than professional, indicating that consumer habits are changing broadly beyond workplace applications [3]. While most economic analysis of AI has focused on productivity impacts in paid work, the research suggests the impact on non-work activities (home production) may be on a similar or larger scale.

Work usage patterns show distinct demographic characteristics. Educated users and those in highly-paid professional occupations are substantially more likely to use ChatGPT for work-related tasks. When examining work-related usage, Writing dominates, accounting for 40% of work-related messages as of June 2025, highlighting chatbots' unique ability to generate and modify digital outputs compared to traditional search engines [4].

### Surprising Findings: What ChatGPT Isn't Primarily Used For

The research challenges several popular assumptions about AI usage. Computer programming accounts for only 4.2% of consumer conversations, contradicting widespread belief that ChatGPT is primarily a coding tool [2][3]. This contrasts sharply with other studies showing 33% of work-related Claude conversations involve programming, suggesting either platform-specific differences or the dominance of non-technical users on ChatGPT's consumer platform.

Self-expression represents an even smaller usage category, with only 2.4% of messages involving relationships and personal reflection (1.9%) or games and role-playing (0.4%) [4]. This suggests that despite media attention on AI companions and creative applications, practical utility dominates actual usage patterns.

## Trends and Evolutionary Patterns

### The Fundamental Shift: From Work to Personal Use

The evolution from 53% to 70% non-work usage represents more than a statistical shift—it indicates a fundamental change in how AI tools are being integrated into daily life. Both work and non-work messages have grown continuously, but non-work messages have grown faster, suggesting ChatGPT is finding product-market fit in personal applications that may not have been anticipated at launch [4].

This trend has significant economic implications. The research cites evidence that U.S. users would need to be paid approximately $98 to give up generative AI for a month, implying at least $97 billion in annual consumer surplus for 2024 alone [2]. This substantial consumer value creation occurs largely outside traditional economic measurements focused on workplace productivity.

### User Intent Evolution: From Doing to Asking

The study introduces a novel taxonomy classifying user intent into three categories: Asking (seeking advice and decision support), Doing (producing outputs or performing tasks), and Expressing (sharing views or feelings). The evolution of these categories reveals changing user relationships with AI [4].

In July 2024, usage was evenly split between Asking and Doing, with under 8% classified as Expressing. By June 2025, the distribution had shifted to 51.6% Asking, 34.6% Doing, and 13.8% Expressing. The growth in Asking messages is particularly significant as these are "consistently rated as having higher quality" than other categories [2][4].

This shift toward decision support rather than task completion suggests users increasingly view ChatGPT as a thinking partner rather than a productivity tool. At work, Doing remains more common at 56%, but even professional usage shows growing emphasis on consultation and advice-seeking [2].

### Work Activity Patterns Across Occupations

The research reveals remarkable consistency in work-related ChatGPT usage across different occupations. Approximately 81% of work-related messages associate with two broad activities: obtaining, documenting, and interpreting information; and making decisions, giving advice, solving problems, and thinking creatively [4].

Most significantly, the work activities "Getting Information" and "Making Decisions and Solving Problems" appear in the top five message frequencies across nearly all occupations, from management and business to STEM to administrative and sales roles. This consistency suggests ChatGPT provides value through fundamental cognitive support activities that transcend specific professional domains [4].

Information-seeking and decision support emerge as the most common ChatGPT use cases across most jobs, reinforcing the platform's role as a decision support system rather than a task automation tool. This finding has important implications for understanding AI's economic value, which the research concludes comes primarily through decision support, especially valuable in knowledge-intensive occupations [4].

### Privacy-Preserving Research Methodology and Reliability

The study employed unprecedented privacy protection measures that set new standards for large-scale AI usage research. All message content analysis was performed via automated LLM-based classifiers on de-identified and PII-scrubbed data, ensuring no human researchers viewed actual message content [4]. The classification system used controlled label spaces and was validated against human judgments using the publicly available WildChat dataset.

The Data Clean Room implementation prevented direct access to user-level demographic data, with strict aggregation limits requiring minimum thresholds of 100 users for any reported statistic. This methodology enables insights into usage patterns while maintaining user privacy—a critical consideration for research involving personal AI interactions [4].

The automated classification system achieved high reliability by incorporating conversation context (the prior 10 messages) and using advanced models (gpt-5-mini for most classifications, gpt-5 for interaction quality assessments). This comprehensive approach provides confidence in the study's findings while establishing methodological precedents for future AI usage research [4].

### Sources

[1] How People Use ChatGPT - by David Deming - Forked Lightning: https://forklightning.substack.com/p/how-people-use-chatgpt

[2] What Is ChatGPT Used For In 2025, Proven NBER Insights: https://binaryverseai.com/what-is-chatgpt-used-for/

[3] ChatGPT Study: 1 In 4 Conversations Now Seek Information: https://www.searchenginejournal.com/chatgpt-study-1-in-4-conversations-now-seek-information/556104/

[4] How People Use ChatGPT | NBER: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf


Research workflow completed!
⏱️  Time: 333.6 seconds
✅ baseline completed

Testing: more_parallel
----------------------------------------
Starting research workflow...


Node: clarify_with_user

I have all the information needed to analyze the NBER working paper "How People Use ChatGPT" and provide insights on the three key areas you've requested:

1. Main findings about how people are using AI (specifically ChatGPT)
2. Most common use cases identified in the research
3. Trends and patterns that emerge from the data

The document provides comprehensive data from ChatGPT's launch in November 2022 through July 2025, including usage patterns, demographic trends, work vs. non-work usage, and detailed classification of conversation types. I will now begin analyzing this research to provide you with detailed insights on these areas.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER Working Paper "How People Use ChatGPT" (Working Paper No. 34

# Comprehensive Analysis of NBER Working Paper "How People Use ChatGPT"

## Main Findings About ChatGPT Usage Patterns and Demographics

The NBER Working Paper 34255 reveals unprecedented adoption patterns for ChatGPT, reaching approximately 10% of the world's adult population by July 2025, with 700 million users sending 18 billion messages weekly [1]. This represents the fastest global diffusion of any new technology in recorded history, reaching 1 million users by December 5, 2022, and 100 million weekly active users by November 2023 [4].

### Adoption Rates and Growth Trajectory

ChatGPT's growth has been exponential, with weekly active users doubling every 7-8 months while message volume grew 5.8x in the past year alone [4]. By June 2025, users were sending more than 2.6 billion messages per day—over 30,000 messages per second [4]. The platform reached critical milestones rapidly: 100 million weekly active users after one year and 350 million after two years, compared to Google Search which took eight years to reach 1 billion daily searches after its 1999 launch [4][6].

### Demographic Evolution and Closing Gaps

The research documents significant demographic shifts over the study period. Initially, early adopters were disproportionately male, with over 80% of users having typically male names [4]. However, the gender gap has closed dramatically—by July 2025, 52% of active users had typically female names, indicating the gender divide has been eliminated [4][5].

Age distribution shows that nearly half of all messages come from users under 26, highlighting ChatGPT's particular resonance with younger demographics [2][6]. Geographic adoption patterns reveal substantial democratization, with higher growth rates in lower-income countries. Middle-income countries experienced 5-6x growth compared to 3x growth in the richest countries [4]. By May 2025, adoption growth rates in the lowest income countries were over 4x those in the highest income countries [5]. Countries like Brazil ($10k GDP per capita), South Korea ($34k), and the United States ($86k) now show similar usage rates despite vastly different economic conditions [4].

### Behavioral Changes and Usage Evolution

User engagement patterns demonstrate increasing intensity over time across all cohorts. Early adopters from Q1 2023 were sending 40% more messages daily by July 2025 than they did two years earlier, while users who joined in late 2024 nearly doubled their message frequency since starting [4]. This pattern suggests both improvements in ChatGPT's capabilities and users discovering new applications for existing features [6].

## Most Common Use Cases and Dominant Categories

The research employed automated classifiers to analyze over 1 million conversations using privacy-preserving methods, revealing clear patterns in how people use ChatGPT [2]. The analysis identified three dominant categories that collectively account for nearly 80% of all conversations.

### The Three Primary Use Case Categories

**Practical Guidance (29%)** represents the largest single category, encompassing tutoring and teaching, how-to advice across various topics, and creative ideation [2]. This category has remained remarkably stable at roughly 29% of overall usage throughout the study period [6]. The consistency suggests this represents a fundamental value proposition that users consistently find valuable.

**Seeking Information (24%)** includes searching for facts, current events, product information, and recipes, appearing to serve as a close substitute for traditional web search [2]. This category has shown significant growth, expanding from 14% to 24% of all usage over the study period [6], indicating users increasingly view ChatGPT as an information retrieval tool.

**Writing (24%)** encompasses automated production of emails and documents, as well as editing, critiquing, summarizing, and translating user-provided text [2]. Notably, this category has declined from 36% of all usage in July 2024 to 24% a year later [6]. Despite this relative decline, Writing remains crucial for work-related activities, accounting for 40% of work-related messages in June 2025 [2]. Importantly, about two-thirds of all Writing messages involve modifying existing user text rather than creating entirely new content [2][6].

### User Intent Classification Framework

Beyond topic categories, the research classified messages by user intent into three types:

- **Asking (49%)**: Seeking information or advice, representing the largest and fastest-growing category, demonstrating users value ChatGPT primarily as an advisor [5]
- **Doing (40%)**: Requesting task completion, dominating work-related usage at 56% of professional queries [2]  
- **Expressing (11%)**: Social interaction and personal reflection [2]

### Specialized Use Cases and Surprising Findings

Education emerged as a major application, with 10.2% of all messages requesting tutoring or teaching, suggesting ChatGPT serves as a significant educational resource [2]. However, several anticipated use cases proved smaller than expected. Computer programming accounts for just 4.2% of messages [2][6], while social and emotional use cases represent only 1.9% of usage [2][6]. Commercial queries represent merely 2% of total volume, compared to approximately 15% of Google searches having commercial intent [6].

## Trends and Patterns in Usage Evolution

### Work vs Non-Work Usage Transformation

The most significant trend identified in the research is the dramatic shift in work versus non-work usage patterns. Non-work messages grew from 53% in June 2024 to over 73% in June 2025, while work-related usage declined proportionally from 47% to 27% [1][2][6]. This shift primarily reflects changing usage patterns within existing user cohorts rather than compositional changes in the user base.

Work usage patterns vary significantly by demographics, being more common among educated users in highly-paid professional occupations [1][2]. For work-related tasks, Writing dominates at 40% of messages, with about 81% of work-related conversations involving information gathering and decision-making support [2]. The research concludes that ChatGPT provides economic value primarily through decision support, particularly in knowledge-intensive jobs [1][2][3].

### Geographic and Economic Democratization Trends  

The data reveals significant democratization trends across geographic and economic lines. Usage gaps between countries have narrowed dramatically, with disproportionate growth in low to middle-income countries ($10,000–40,000 GDP per capita) between May 2024 and May 2025 [6]. This pattern suggests AI tools are becoming more accessible globally, potentially reducing digital divides rather than exacerbating them.

### Evolution Within User Cohorts

All user cohorts show similar usage evolution patterns—relatively flat growth through 2024 but substantial increases beginning in early 2025 [4]. This timing suggests significant improvements in ChatGPT's capabilities and user-friendliness influenced adoption patterns across all demographics. Earlier adopters consistently maintain higher usage levels while also showing continued growth, indicating both user learning effects and platform improvements [6].

### Economic Impact Beyond Traditional Productivity Measures

The research reveals that while economic analysis of AI typically focuses on workplace productivity, ChatGPT's impact on non-work activities (home production) occurs on a similar or potentially larger scale [2][6]. This finding aligns with research by Collis and Brynjolfsson (2025) estimating consumer surplus of at least $97 billion in 2024 alone in the US [2]. The study suggests ChatGPT generates value that traditional measures like GDP fail to capture [5].

The analysis challenges fundamental assumptions about AI integration into society, showing people use AI tools differently than experts anticipated [7]. Contrary to predictions emphasizing work automation and professional productivity, the research demonstrates that 73% of ChatGPT interactions are personal rather than professional [7], representing the most comprehensive analysis of actual consumer AI usage ever conducted [5].

### Sources

[1] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255
[2] [PDF] How People Use ChatGPT - National Bureau of Economic Research: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf
[3] How People Use ChatGPT - SSRN: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5487080
[4] How People Use ChatGPT - by David Deming - Forked Lightning: https://forklightning.substack.com/p/how-people-use-chatgpt
[5] How people are using ChatGPT | OpenAI: https://openai.com/index/how-people-are-using-chatgpt/
[6] How people use ChatGPT and its implications, Portfolio Change: https://www.mbi-deepdives.com/how-people-use-chatgpt-and-its-implications-portfolio-change/
[7] Harvard AI Usage Study: How 700 Million People Really Use ChatGPT: https://medium.com/write-a-catalyst/harvard-ai-usage-study-how-700-million-people-really-use-chatgpt-7df863a2a374


Research workflow completed!
⏱️  Time: 220.7 seconds
✅ more_parallel completed

Testing: deeper_research
----------------------------------------
Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with analyzing the NBER working paper "How People Use ChatGPT" by Chatterji et al. (2025). Based on your request, I will provide insights on: 1) the main findings about how people are using AI/ChatGPT, 2) the most common use cases identified in the study, and 3) key trends and patterns that emerge from the data. The document contains comprehensive usage data from ChatGPT's consumer product from November 2022 through July 2025, including classification of conversation types, work vs. non-work usage patterns, and demographic insights. I will now begin analyzing this research to provide you with a detailed report on these three key areas.

Node: write_research_brief

Research Brief Generated:
I need a comprehensive analysis of the NBER working pape


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Comprehensive Analysis: How People Use ChatGPT - NBER Working Paper Findings

## 1. Main Findings About AI/ChatGPT Usage Patterns

### Unprecedented Global Adoption Scale

ChatGPT's growth trajectory represents the fastest global technology diffusion in history. Launched in November 2022, ChatGPT reached 1 million users by December 5, 2022 (just 5 days after launch) and hit 100 million weekly active users by November 2023. By July 2025, the platform had been adopted by approximately 700 million users worldwide, representing around 10% of the global adult population [1][3][5].

The scale of usage is staggering: by June 2025, users were sending over 18 billion messages per week, equivalent to more than 2.6 billion messages daily or over 30,000 messages per second [1][2]. Total message volume increased 5.8 times between July 2024 and July 2025 alone, with weekly active users doubling every 7-8 months [2][14].

### Dramatic Demographic Evolution

**Gender Gap Closure**: The most striking demographic finding is the complete reversal of the gender gap. Early adopters were overwhelmingly male, with more than 80% of initial users having typically masculine names [1][2][6][8]. However, this disparity has narrowed dramatically over time. By January 2024, 37% of users had typically feminine names, and by July 2025, this figure reached 52% [6][8]. This indicates the gender gap in ChatGPT usage may have closed completely, with female users now representing a slight majority [1].

**Age Distribution**: Nearly half (46%) of all adult messages come from users between 18 to 25 years of age, though age gaps have narrowed somewhat in recent months [3][5][6][14]. Older users demonstrate different usage patterns, being more likely to send work-related messages. Work-related messages comprised approximately 23% of messages for users under age 26, with this share increasing with age, except for users 66 and older, who show only 16% work-related usage [7].

**Geographic and Economic Patterns**: ChatGPT adoption has grown relatively faster in low- and middle-income countries. As of May 2025, adoption growth rates in the lowest-income countries were four times higher than those in the highest-income countries [5][8]. Countries like Brazil, South Korea, and the United States now have similar ChatGPT usage rates despite vastly different GDP per capita levels ($10k, $34k, and $86k respectively) [2].

### Educational and Professional Usage Patterns

Educated users and those in highly-paid professional occupations are substantially more likely to use ChatGPT for work purposes [3][5][7]. About 48% of graduate degree holders' messages are work-related versus 37% for those with less than a bachelor's degree [1]. Highly educated professionals are disproportionately likely to use ChatGPT for "asking" rather than "doing" tasks, focusing more on guidance and decision support [1].

## 2. Most Common Use Cases Analysis

### Three Dominant Categories

The study reveals that nearly 80% of all ChatGPT conversations fall into three primary categories: "Practical Guidance," "Seeking Information," and "Writing" [3][5][7]. These categories collectively account for 78% of all user conversations [4].

### Practical Guidance (28-29%)

Practical Guidance emerges as the most common use case, accounting for 28.3% of all messages [2][7][14]. This category has remained remarkably consistent at roughly 29% of overall usage throughout the study period [14]. The category encompasses:

- **Tutoring and Teaching**: About 36% of Practical Guidance messages are requests for educational support, representing 10.2% of all ChatGPT usage [7]
- **How-to Advice**: Approximately 30% of this category consists of general how-to guidance across various topics [7]
- **Creative Ideation**: Users frequently seek assistance with brainstorming and creative problem-solving [3][5][7]

This finding highlights ChatGPT's role as a digital mentor and problem-solving companion, providing personalized guidance across diverse domains.

### Seeking Information (21-24%)

The Seeking Information category has shown significant growth, expanding from 14% to 24% of all usage between July 2024 and July 2025, currently accounting for 21.3% of all messages [2][7][14]. This category includes:

- **Factual Queries**: Users search for information about people, current events, and general knowledge
- **Product Information**: Research and comparison of products and services
- **Current Events**: Real-time information seeking about news and developments
- **Recipes and Practical Information**: Everyday information needs [3][5][7]

This usage pattern suggests ChatGPT serves as a close substitute for traditional web search, offering conversational and contextual information retrieval.

### Writing (24-28%)

Writing accounts for 28.1% of all conversations but has shown a decline from 36% in July 2024 to 24% a year later [2][7][14]. This category demonstrates the platform's unique capability to generate and manipulate digital content, including:

- **Text Modification**: About two-thirds of all Writing messages ask ChatGPT to modify user-provided text rather than creating new content from scratch [3][5][7]
- **The five sub-categories within Writing (in order of frequency)**:
  - Editing or Critiquing Provided Text
  - Personal Writing or Communication
  - Translation
  - Argument or Summary Generation
  - Writing Fiction [7]

### Work-Specific Use Cases

Writing dominates workplace usage, accounting for 40-42% of work-related messages as of June 2025 [3][5][7]. Nearly 35% of all work-related queries are "Doing" messages related to Writing tasks [7]. This finding underscores ChatGPT's unique value proposition compared to traditional search engines: the ability to generate actionable digital outputs rather than just provide information.

Practical Guidance represents the second most common work use case at 24% [7]. The study found that 58% of work-related messages are associated with two broad work activities: obtaining, documenting, and interpreting information; and making decisions, giving advice, solving problems, and thinking creatively [1].

### Less Common But Notable Use Cases

**Computer Programming**: Contrary to popular perception, computer programming accounts for only 4.2% of consumer conversations, significantly less than competing AI tools like Claude (which shows 33% programming-related conversations) [2][3][5][7].

**Technical Help**: This category has declined from 12% of all usage in July 2024 to around 5% a year later [7].

**Social and Emotional Support**: The share of messages related to companionship or social-emotional issues remains relatively small, with only 1.9% of messages focused on Relationships and Personal Reflection and 0.4% related to Games and Role Play [7].

**Self-Expression**: Accounts for only 2.4% of all ChatGPT messages [7].

## 3. Trends and Patterns from the Data

### Fundamental Shift from Work to Non-Work Usage

The most significant trend identified in the study is the dramatic shift from work-related to personal usage. In June 2024, usage was split between 53% non-work and 47% work messages. By June 2025, this had shifted dramatically to 73% non-work and 27% work usage [4][7].

Specifically, the data shows:
- **June 2024**: 238 million daily non-work messages (53%) and 213 million work messages (47%), totaling 451 million daily messages
- **June 2025**: 1,911 million daily non-work messages (73%) and 716 million work messages (27%), totaling 2,627 million daily messages [7]

Crucially, this shift is primarily due to changing usage patterns within existing user cohorts rather than changes in the composition of new users [14]. Both work and non-work messages have grown continuously, but non-work messages have grown faster [7].

### Cohort Analysis and User Engagement Evolution

All user signup cohorts show similar patterns: relatively flat usage through most of 2024, followed by substantial increases beginning in late 2024/early 2025 [1]. Early adopters from Q1 2023 were sending 40% more messages per day by July 2025 than they did two years earlier [1].

Each cohort demonstrates that earlier sign-ups consistently maintain higher usage levels, but usage has grown within every cohort. The researchers interpret this as resulting from both improvements in model capabilities and users gradually discovering new applications for existing features [7][14].

### User Intent Evolution: From Doing to Asking

The study reveals a significant shift in how users interact with ChatGPT through the "Asking/Doing/Expressing" framework:

- **July 2024**: Usage was evenly split between Asking and Doing, with under 8% Expressing
- **June 2025**: 51.6% Asking, 34.6% Doing, and 13.8% Expressing [7]

This trend toward "Asking" messages (seeking guidance, advice, or information) over "Doing" messages (completing specific tasks) suggests users increasingly value ChatGPT as a conversational partner and advisor rather than merely a task completion tool [2].

### Geographic and Economic Democratization

Middle-income countries have shown 5-6x growth in usage compared to 3x growth in the richest countries [2]. Growth has been strongest in low- and middle-income countries ($10,000–$40,000 GDP-per-capita) over the last year [1][14]. This trend suggests ChatGPT is becoming a democratizing technology, providing advanced AI capabilities to users regardless of their economic circumstances.

### Quality and User Satisfaction Improvements

"Asking" messages have shown faster growth over the last year and are rated as having higher quality both by user satisfaction classifiers and direct user feedback [7]. Positive user ratings ("good" interactions) are now over four times more common than negative ones [1], indicating substantial improvements in user experience and model performance.

### Multimedia Usage Growth

Multimedia usage grew from 2% to just over 7% of total usage, with a notable spike in April 2025 following ChatGPT's release of new image-generation capabilities [7]. While this spike later attenuated, the elevated usage level has persisted, suggesting multimedia capabilities are becoming an integral part of the user experience [5].

### Economic Value and Consumer Surplus

The study estimates that U.S. users would need to be paid roughly $98 to give up generative AI for a month, implying at least $97 billion in annual consumer surplus in 2024 [2][8]. This finding aligns with research by Collis and Brynjolfsson (2025) and underscores the substantial economic value users derive from ChatGPT, particularly for non-work activities.

### Professional Work Activity Mapping

Using O*NET analysis, the study found that 81% of work-related messages are associated with two broad work activities: obtaining, documenting, and interpreting information; and making decisions, giving advice, solving problems, and thinking creatively [7]. The work activities "Getting Information" and "Making Decisions and Solving Problems" appear in the top five message frequencies across nearly all occupations, from management and business to STEM to administrative and sales [7].

This finding suggests ChatGPT's value proposition transcends specific job categories, providing decision support and information processing capabilities that are universally valuable across different types of professional work.

### Sources

[1] How People Use ChatGPT - by David Deming - Forked Lightning: https://forklightning.substack.com/p/how-people-use-chatgpt

[2] What Is ChatGPT Used For In 2025, Proven NBER Insights: https://binaryverseai.com/what-is-chatgpt-used-for/

[3] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255

[4] How People Use ChatGPT | NBER: https://www.kif.re.kr/kif4/publication/redirect?mid=13&cno=353312

[5] OpenAI study on ChatGPT usage trends: 24% messages seek info, majority of chats personal: https://indianexpress.com/article/technology/artificial-intelligence/openai-study-chatgpt-usage-trends-key-insights-data-10253385/

[6] OpenAI releases research on ChatGPT usage worldwide - LinkedIn: https://www.linkedin.com/posts/aaron-ronnie-chatterji_this-morning-the-openai-economic-research-activity-7373377911476649986-_n_s

[7] How People Use ChatGPT: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf

[8] How people are using ChatGPT: https://openai.com/index/how-people-are-using-chatgpt/

[14] How people use ChatGPT and its implications, Portfolio: https://www.mbi-deepdives.com/how-people-use-chatgpt-and-its-implications-portfolio-change/


Research workflow completed!
⏱️  Time: 351.6 seconds
✅ deeper_research completed

All tests completed!


Best for Speed: More Parallel configuration
- Use when you need quick results
- Good for time-sensitive research

Best for Thoroughness: Deeper Research configuration
- Use when you need comprehensive analysis
- Accept longer execution time for better quality

Baseline: Middle ground
- Balanced approach
- Good default setting

## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs